#MapReduce
MapReduce is a batch-processing framework where data is read from disk, processed in two phases (Map and Reduce), and written back to disk between each stage.

Limitations of MapReduce:-
1. Every intermediate result is written to disk, causing high I/O latency
2. Poorly suited for iterative tasks like machine learning
3. No support for real-time or interactive querying
4. Complex pipelines need multiple MapReduce jobs, multiplying the overhead

# Apache Spark
Apache Spark is an open-source, distributed computation engine built for speed and flexibility, capable of handling batch, streaming, and machine learning workloads within a single framework.

##Advantages of Apache Spark:-
1. In-Memory Computation: Keeps intermediate data in RAM instead of writing to disk, drastically cutting down execution time.
2. Speed: Outperforms MapReduce by up to 100x for certain workloads due to in-memory processing.
3. Multilingual Support: Provides APIs in Python, Scala, Java, and R.
4. Resilience: Tracks data lineage through RDDs to automatically rebuild lost partitions.
5. Versatility: Handles SQL, streaming, ML, and graph processing under one unified engine.

#Spark DataFrames and Immutability
A DataFrame is a two-dimensional, schema-aware data structure distributed across a cluster, where data is organised into typed columns — much like a spreadsheet backed by a parallel processing engine.

###Characteristics of DataFrames
1. Each column has a fixed name and data type defined by a schema.
2. Query execution is optimised internally by Spark's Catalyst Optimizer.
3. Supports both the programmatic DataFrame API and raw SQL via spark.sql().
4. Designed to scale from a single machine to thousands of nodes.

###Immutability
Once a DataFrame is created, it cannot be altered. Any operation on it produces a brand-new DataFrame, leaving the original intact.

Example:

df_filtered = df.filter(df.rating > 4.0)

Here df stays untouched. Spark records the transformation in its execution plan and materialises df_filtered only when an action is triggered.

In [49]:
from google.colab import files

uploaded = files.upload()

Saving amazon.csv to amazon (3).csv


In [50]:
!pip install pyspark -q

In [51]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Amazon_Product_Analysis") \
    .getOrCreate()

In [52]:
# Load with header=True, all columns come in as strings to avoid inferSchema issues
df = spark.read.csv(
    "amazon.csv",
    header=True,
    inferSchema=False
)

df.show(5)

+----------+--------------------+--------------------+----------------+------------+-------------------+------+------------+---+-------+
|product_id|        product_name|            category|discounted_price|actual_price|discount_percentage|rating|rating_count|age| region|
+----------+--------------------+--------------------+----------------+------------+-------------------+------+------------+---+-------+
|B00VA7YYUO|Apsara Platinum P...|Home&Kitchen|Craf...|             ₹99|         ₹99|                 0%|   4.3|       5,036| 31|   East|
|B014I8SX4Y|Amazon Basics Hig...|Electronics|HomeT...|            ₹309|      ₹1,400|                78%|   4.4|    4,26,973| 47|Central|
|B06XFTHCNY|CableCreation RCA...|Electronics|HomeT...|            ₹439|        ₹758|                42%|   4.2|       4,296| 45|Central|
|B09QS9X9L8|Redmi Note 11 (Ho...|Electronics|Mobil...|         ₹12,999|     ₹18,999|                32%|   4.1|      50,772| 46|   West|
|B01FSYQ2A4|boAt Rockerz 400 ...|Electron

In [53]:
df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- discounted_price: string (nullable = true)
 |-- actual_price: string (nullable = true)
 |-- discount_percentage: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- rating_count: string (nullable = true)
 |-- age: string (nullable = true)
 |-- region: string (nullable = true)



In [54]:
print("Total Rows:", df.count())

Total Rows: 1495


In [55]:
# Check duplicate product entries
df.groupBy("product_id") \
  .count() \
  .filter("count > 1") \
  .show()

+----------+-----+
|product_id|count|
+----------+-----+
|B09T2WRLJJ|    2|
|B00NH11KIK|    2|
|B0BFWGBX61|    2|
|B01C8P29N0|    2|
|B0B5LVS732|    2|
|B01M4GGIVU|    2|
|B0B9BXKBC7|    2|
|B07P681N66|    2|
|B097R25DP7|    2|
|B0BMXMLSMM|    2|
|B09NVPSCQT|    2|
|B0B86CDHL1|    2|
|B09YV463SW|    2|
|B002PD61Y4|    2|
|B09MT84WV5|    2|
|B09F6S8BT6|    2|
|B0B3RRWSF6|    2|
|B08CDKQ8T6|    2|
|B09TBCVJS3|    2|
|B09Z6WH2N1|    2|
+----------+-----+
only showing top 20 rows


In [56]:
print("Before:", df.count())

df = df.dropDuplicates()

print("After:", df.count())

Before: 1495
After: 1458


In [57]:
from pyspark.sql.functions import col, count, when, trim, regexp_replace, sum, avg, min, max

# Count nulls in each column
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+----------+------------+--------+----------------+------------+-------------------+------+------------+---+------+
|product_id|product_name|category|discounted_price|actual_price|discount_percentage|rating|rating_count|age|region|
+----------+------------+--------+----------------+------------+-------------------+------+------------+---+------+
|         0|           0|       0|               0|           0|                  0|    38|           2| 38|     2|
+----------+------------+--------+----------------+------------+-------------------+------+------------+---+------+



In [58]:
df = df.fillna({
    "age": "0",
    "rating": "0",
    "rating_count": "0",
    "region": "Unknown"
})
df.orderBy("product_id").show(10)

+----------+--------------------+--------------------+----------------+------------+-------------------+------+------------+---+-------+
|product_id|        product_name|            category|discounted_price|actual_price|discount_percentage|rating|rating_count|age| region|
+----------+--------------------+--------------------+----------------+------------+-------------------+------+------------+---+-------+
|B002PD61Y4|D-Link DWA-131 30...|Computers&Accesso...|            ₹507|      ₹1,208|                58%|   4.1|       8,131| 43|   East|
|B002PD61Y4|D-Link DWA-131 30...|Computers&Accesso...|            ₹507|      ₹1,208|                58%|     0|       8,131|  0|   West|
|B002SZEOLG|TP-Link Nano USB ...|Computers&Accesso...|            ₹749|      ₹1,339|                44%|   4.2|    1,79,692| 36|  South|
|B003B00484|Duracell Plus AAA...|Electronics|Gener...|            ₹399|        ₹499|                20%|   4.3|      27,201| 40|Central|
|B003L62T7W|Logitech B100 Wir...|Computer

In [59]:
# Remove rows where product_name is blank
df = df.filter(trim(col("product_name")) != "")
df.orderBy("product_id").show(10)

+----------+--------------------+--------------------+----------------+------------+-------------------+------+------------+---+-------+
|product_id|        product_name|            category|discounted_price|actual_price|discount_percentage|rating|rating_count|age| region|
+----------+--------------------+--------------------+----------------+------------+-------------------+------+------------+---+-------+
|B002PD61Y4|D-Link DWA-131 30...|Computers&Accesso...|            ₹507|      ₹1,208|                58%|   4.1|       8,131| 43|   East|
|B002PD61Y4|D-Link DWA-131 30...|Computers&Accesso...|            ₹507|      ₹1,208|                58%|     0|       8,131|  0|   West|
|B002SZEOLG|TP-Link Nano USB ...|Computers&Accesso...|            ₹749|      ₹1,339|                44%|   4.2|    1,79,692| 36|  South|
|B003B00484|Duracell Plus AAA...|Electronics|Gener...|            ₹399|        ₹499|                20%|   4.3|      27,201| 40|Central|
|B003L62T7W|Logitech B100 Wir...|Computer

In [60]:
df = df.withColumnRenamed(
    "product_name",
    "product_full_name"
)
df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_full_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- discounted_price: string (nullable = true)
 |-- actual_price: string (nullable = true)
 |-- discount_percentage: string (nullable = true)
 |-- rating: string (nullable = false)
 |-- rating_count: string (nullable = false)
 |-- age: string (nullable = false)
 |-- region: string (nullable = false)



In [61]:
from pyspark.sql.functions import col, regexp_replace, when

def safe_int(c):
    cleaned = regexp_replace(col(c), "[^0-9]", "")
    return when(cleaned == "", None).otherwise(cleaned.cast("int"))

def safe_float(c):
    cleaned = regexp_replace(col(c), "[^0-9.]", "")
    return when(cleaned == "", None).otherwise(cleaned.cast("float"))

df = df \
    .withColumn("actual_price",      safe_float("actual_price")) \
    .withColumn("discounted_price",  safe_float("discounted_price")) \
    .withColumn("discount_percentage", safe_int("discount_percentage")) \
    .withColumn("rating_count",      safe_int("rating_count")) \
    .withColumn("rating",            safe_float("rating")) \
    .withColumn("age",               safe_int("age"))

df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_full_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- discounted_price: float (nullable = true)
 |-- actual_price: float (nullable = true)
 |-- discount_percentage: integer (nullable = true)
 |-- rating: float (nullable = true)
 |-- rating_count: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- region: string (nullable = false)



In [62]:
# Customers in the 20-40 age bracket
df.filter(
    (col("age") >= 20) &
    (col("age") <= 40)
).show()

+----------+--------------------+--------------------+----------------+------------+-------------------+------+------------+---+-------+
|product_id|   product_full_name|            category|discounted_price|actual_price|discount_percentage|rating|rating_count|age| region|
+----------+--------------------+--------------------+----------------+------------+-------------------+------+------------+---+-------+
|B098JYT4SY|Zebronics Zeb-Jag...|Computers&Accesso...|           399.0|      1190.0|                 66|   4.1|        2809| 38|  South|
|B09YV3K34W|Fire-Boltt India'...|Electronics|Weara...|          2199.0|      9999.0|                 78|   4.2|       29472| 28|   East|
|B092X94QNQ|boAt Rockerz 330 ...|Electronics|Headp...|          1499.0|      3990.0|                 62|   4.1|      109864| 27|   West|
|B08BCKN299|Sounce Gold Plate...|Electronics|Headp...|           120.0|       999.0|                 88|   3.9|        6491| 38|  North|
|B08243SKCK|Vedini Transparen...|Home&Kit

In [63]:
# Products belonging to Electronics
df.filter(
    col("category").startswith("Electronics")
).show()

+----------+--------------------+--------------------+----------------+------------+-------------------+------+------------+---+-------+
|product_id|   product_full_name|            category|discounted_price|actual_price|discount_percentage|rating|rating_count|age| region|
+----------+--------------------+--------------------+----------------+------------+-------------------+------+------------+---+-------+
|B09WRMNJ9G|OnePlus 10R 5G (F...|Electronics|Mobil...|         34999.0|     38999.0|                 10|   4.2|       11029| 41|  South|
|B07WJWRNVK|iQOO vivo Z6 5G (...|Electronics|Mobil...|         16499.0|     20990.0|                 21|   4.0|       21350| 52|   West|
|B09YV3K34W|Fire-Boltt India'...|Electronics|Weara...|          2199.0|      9999.0|                 78|   4.2|       29472| 28|   East|
|B092X94QNQ|boAt Rockerz 330 ...|Electronics|Headp...|          1499.0|      3990.0|                 62|   4.1|      109864| 27|   West|
|B0B217Z5VK|Noise Buds VS402 ...|Electron

In [64]:
# Orders from the North region
df.filter(
    col("region") == "North"
).show()

+----------+--------------------+--------------------+----------------+------------+-------------------+------+------------+---+------+
|product_id|   product_full_name|            category|discounted_price|actual_price|discount_percentage|rating|rating_count|age|region|
+----------+--------------------+--------------------+----------------+------------+-------------------+------+------------+---+------+
|B08BCKN299|Sounce Gold Plate...|Electronics|Headp...|           120.0|       999.0|                 88|   3.9|        6491| 38| North|
|B08243SKCK|Vedini Transparen...|Home&Kitchen|Home...|           189.0|       299.0|                 37|   4.2|        2737| 30| North|
|B07ZR4S1G4|Universal Remote ...|Electronics|HomeT...|           239.0|       699.0|                 66|   4.4|        2640| 26| North|
|B09Q3M3WLJ|Robustrion [Anti-...|Computers&Accesso...|           399.0|      1499.0|                 73|   4.0|         691| 33| North|
|B09RFC46VP|Redmi 108 cm (43 ...|Electronics|Hom

In [65]:
df.select(
    count("*").alias("total_records"),
    sum("actual_price").alias("total_revenue"),
    avg("actual_price").alias("avg_price"),
    min("actual_price").alias("min_price"),
    max("actual_price").alias("max_price")
).show()

+-------------+-----------------+-----------------+---------+---------+
|total_records|    total_revenue|        avg_price|min_price|max_price|
+-------------+-----------------+-----------------+---------+---------+
|         1458|8709425.279907227|6056.623977682355|      2.0| 240280.0|
+-------------+-----------------+-----------------+---------+---------+



#Wide Transformations and Shuffle Operations
Spark splits transformations into two categories — narrow and wide. Narrow transformations (like filter and select) work independently on each partition with no data movement. Wide transformations require Spark to collect and redistribute data across partitions, a process known as a shuffle.

groupBy(), join(), and distinct() all fall under wide transformations.
###For example:

df.groupBy("region").count().show()

To count products per region, Spark must pull all rows sharing a region into the same partition. This cross-partition data movement (shuffle) is necessary but costly — it adds network transfer time and temporary disk writes, which is why minimising unnecessary groupBy operations matters in large-scale pipelines.

In [66]:
df.groupBy("region") \
  .agg(
      sum("actual_price").alias("total_revenue")
  ) \
  .show()

+--------------------+------------------+
|              region|     total_revenue|
+--------------------+------------------+
|              ₹1,999|             459.0|
|                 4.2|          961120.0|
|                 66%|             543.0|
|                  59|             549.0|
|              ₹1,499|             140.0|
|              22,638|              NULL|
|                 4.4|              NULL|
|                  22|           37247.0|
|              ₹7,990|             200.0|
|Electronics|Weara...|              NULL|
|             ₹15,990|              NULL|
|                  27|              NULL|
|                 4.3|             203.0|
|                  46|             999.0|
|                 79%|              NULL|
|                 4.1|              NULL|
|              ₹1,898|               2.0|
|             Unknown|          240282.0|
|               South|1661050.2799072266|
|                 60%|               4.0|
+--------------------+------------

In [67]:
df.groupBy("category") \
  .agg(
      avg("actual_price").alias("avg_price")
  ) \
  .show()

+--------------------+------------------+
|            category|         avg_price|
+--------------------+------------------+
|           reminders|              NULL|
|Computers&Accesso...|             799.0|
|OfficeProducts|Of...|             150.0|
|OfficeProducts|Of...|267.85714285714283|
|Electronics|Camer...|             400.0|
|Computers&Accesso...|             999.0|
|      TWS Connection|               2.0|
|Computers&Accesso...|            1519.0|
|Computers&Accesso...|            1399.5|
|     123 Sports Mode|              NULL|
|         81X800LGIN"|           37247.0|
|Computers&Accesso...|3432.3333333333335|
|Health&PersonalCa...|            1900.0|
|Home&Kitchen|Kitc...|1061.6363636363637|
|    170+ Watch Faces|              NULL|
|Home&Kitchen|Kitc...| 580.5714285714286|
|     60 Sports Modes|               2.0|
|OfficeProducts|Of...|            2999.0|
|Home&Kitchen|Heat...|            2910.0|
|Electronics|Mobil...|            1999.0|
+--------------------+------------

In [68]:
# Regions where average price exceeds 5000
df.groupBy("region") \
  .agg(
      avg("actual_price").alias("avg_price")
  ) \
  .filter(col("avg_price") > 5000) \
  .show()

+-------+-----------------+
| region|        avg_price|
+-------+-----------------+
|    4.2|         240280.0|
|     22|          37247.0|
|Unknown|         120141.0|
|  South|6106.802499658921|
|Central| 5218.36303630363|
|   West|6145.041666666667|
|  North|5026.810909090909|
+-------+-----------------+



In [69]:
df.groupBy("region").count().show()

+--------------------+-----+
|              region|count|
+--------------------+-----+
|              ₹1,999|    4|
|                 4.2|    4|
|                 66%|    1|
|                  59|    1|
|              ₹1,499|    1|
|              22,638|    2|
|                 4.4|    1|
|                  22|    1|
|              ₹7,990|    2|
|Electronics|Weara...|    2|
|             ₹15,990|    1|
|                  27|    1|
|                 4.3|    4|
|                  46|    1|
|                 79%|    1|
|                 4.1|    1|
|              ₹1,898|    1|
|             Unknown|    2|
|               South|  272|
|                 60%|    2|
+--------------------+-----+
only showing top 20 rows


In [70]:
# Full pipeline: clean -> group -> aggregate
result = (
    df.dropDuplicates()
      .fillna({"age": 0, "region": "Unknown"})
      .groupBy("category")
      .agg(
          count("*").alias("total_products"),
          sum("actual_price").alias("total_revenue"),
          avg("actual_price").alias("avg_price")
      )
)

result.show()

+--------------------+--------------+-------------+------------------+
|            category|total_products|total_revenue|         avg_price|
+--------------------+--------------+-------------+------------------+
|           reminders|             1|         NULL|              NULL|
|Computers&Accesso...|             1|        799.0|             799.0|
|OfficeProducts|Of...|             4|        600.0|             150.0|
|Electronics|Camer...|             1|        400.0|             400.0|
|OfficeProducts|Of...|             7|       1875.0|267.85714285714283|
|Computers&Accesso...|             1|        999.0|             999.0|
|      TWS Connection|             2|          4.0|               2.0|
|Computers&Accesso...|             5|       7595.0|            1519.0|
|Computers&Accesso...|             2|       2799.0|            1399.5|
|     123 Sports Mode|             1|         NULL|              NULL|
|         81X800LGIN"|             1|      37247.0|           37247.0|
|Compu

#Insights

1. **Data Quality** – The dataset had 42 null values in `rating` and `age` columns, and the `rating` column contained a stray `'|'` value. Price columns stored values as strings with `₹` symbols and commas, requiring cleanup before any numeric operation.

2. **Filtering** – Most products in the dataset fall in the 20–40 age bracket. Electronics is the dominant category, appearing across the majority of records.

3. **Aggregations** – The `actual_price` column showed a wide range between min and max values, indicating the dataset covers both budget and premium products.

4. **GroupBy Results** – Revenue and product count varied across regions. Only certain regions crossed the avg price threshold of 5000, showing regional pricing differences.

5. **Pipeline** – Combining `dropDuplicates`, `fillna`, and `groupBy` in a single chained pipeline demonstrated how Spark's lazy evaluation builds the full execution plan before running — nothing executes until `.show()` is called.